In [0]:
# ============================================================
# Notebook: ml_build_churn_features
# Purpose:  Build customer-level feature table for churn model
# Inputs:   Gold parquet tables in ADLS Gen2
# Output:   Parquet/Delta table: gold_ml/churn_feature_table
# ============================================================

# Data Access

In [0]:
storage_account = "nastorage003"

adls_oauth_configs = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": "<YOUR_AZURE_CLIENT_ID>",
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": "<YOUR_AZURE_CLIENT_SECRET>",
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": "https://login.microsoftonline.com/<YOUR_AZURE_TENANT_ID>/oauth2/token",
}

In [0]:
for key, value in adls_oauth_configs.items():
    spark.conf.set(key, value)

dbutils.fs.ls("abfss://gold@nastorage003.dfs.core.windows.net/")

[FileInfo(path='abfss://gold@nastorage003.dfs.core.windows.net/DimCustomer/', name='DimCustomer/', size=0, modificationTime=1780037919000),
 FileInfo(path='abfss://gold@nastorage003.dfs.core.windows.net/DimDate/', name='DimDate/', size=0, modificationTime=1780037899000),
 FileInfo(path='abfss://gold@nastorage003.dfs.core.windows.net/DimIssueCategory/', name='DimIssueCategory/', size=0, modificationTime=1780037912000),
 FileInfo(path='abfss://gold@nastorage003.dfs.core.windows.net/DimPaymentStatus/', name='DimPaymentStatus/', size=0, modificationTime=1780037916000),
 FileInfo(path='abfss://gold@nastorage003.dfs.core.windows.net/DimRegion/', name='DimRegion/', size=0, modificationTime=1780037908000),
 FileInfo(path='abfss://gold@nastorage003.dfs.core.windows.net/FactBilling/', name='FactBilling/', size=0, modificationTime=1780037932000),
 FileInfo(path='abfss://gold@nastorage003.dfs.core.windows.net/FactTelemetry/', name='FactTelemetry/', size=0, modificationTime=1780037927000),
 FileInf

# Load Data

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window


In [0]:
# -----------------------------
# 1. Configuration
# -----------------------------

# Base path to your Gold container in ADLS Gen2
# Adjust storage account / container names to match your environment
GOLD_BASE_PATH = f"abfss://gold@{storage_account}.dfs.core.windows.net"

# Output path for ML features (you can also use Delta tables instead)
CHURN_FEATURE_PATH = f"{GOLD_BASE_PATH}/churn_feature_table"

In [0]:
#-----------------------------
# 2. Load Gold Dim/Fact tables
# -----------------------------
customers_df = (spark.read
    .format("delta")
    .load(f"{GOLD_BASE_PATH}/DimCustomer"))

telemetry_df = (spark.read
    .format("delta")
    .load(f"{GOLD_BASE_PATH}/FactTelemetry"))

tickets_df = (spark.read
    .format("delta")
    .load(f"{GOLD_BASE_PATH}/FactTickets"))

billing_df = (spark.read
    .format("delta")
    .load(f"{GOLD_BASE_PATH}/FactBilling"))

dim_date_df = (
    spark.read
    .format("delta")
    .load(f"{GOLD_BASE_PATH}/DimDate"))

dim_payment_status_df = (
    spark.read
    .format("delta")
    .load(f"{GOLD_BASE_PATH}/DimPaymentStatus"))

dim_region_df = (
    spark.read
    .format("delta")
    .load(f"{GOLD_BASE_PATH}/DimRegion")
)

In [0]:
display(customers_df)

customer_key,customer_id,customer_type,customer_id_str,region_key,plan_type,contract_type,monthly_charge_aud,tenure_months,has_cybersecurity_addon,has_voip_addon,onboarded_date,churn_flag,churn_date
1,CUST-00001,business,CUST-00001,4,Fibre,month-to-month,307.44,34,true,true,2023-08-02,true,2025-09-29
2,CUST-00002,business,CUST-00002,1,NBN100,1-year,76.98,26,true,true,2024-02-17,true,2025-09-27
3,CUST-00003,residential,CUST-00003,1,NBN100,1-year,76.96,7,false,true,2020-05-02,true,2020-07-30
4,CUST-00004,business,CUST-00004,2,Fibre,2-year,292.89,29,false,true,2021-02-11,false,null
5,CUST-00005,business,CUST-00005,6,NBN1000,month-to-month,120.3,16,false,true,2022-07-08,false,null
6,CUST-00006,business,CUST-00006,3,NBN250,1-year,96.58,15,true,true,2024-07-08,false,null
7,CUST-00007,residential,CUST-00007,4,NBN50,1-year,60.15,33,false,false,2021-03-16,false,null
8,CUST-00008,business,CUST-00008,6,Fibre,1-year,301.08,6,true,true,2021-01-31,false,null
9,CUST-00009,business,CUST-00009,6,NBN100,2-year,79.16,31,true,false,2020-03-29,false,null
10,CUST-00010,business,CUST-00010,1,NBN1000,month-to-month,116.72,3,false,false,2024-03-30,true,2024-05-17


# Create the feature dataset

In [0]:
# -----------------------------
# 3. Customer base features
# -----------------------------

# First select from DimCustomer
cust_base = (
    customers_df
    .select(
        "customer_key",
        "customer_id",
        "customer_type",
        "region_key",
        "plan_type",
        "contract_type",
        "monthly_charge_aud",
        "tenure_months",
        "has_cybersecurity_addon",
        "has_voip_addon",
        "churn_flag",
        "churn_date"
    )
)

# Then join DimRegion to add the human-readable region label
cust_feats = (
    cust_base
    .join(
        dim_region_df.select("region_key", "region"),
        on="region_key",
        how="left"
    )
    .withColumnRenamed("region_enum", "region")
)

# -----------------------------
# 4. Network telemetry features
# -----------------------------

telemetry_feats = (
    telemetry_df
    .groupBy("customer_key")
    .agg(
        F.avg("daily_bandwidth_gb").alias("avg_daily_bandwidth_gb"),
        F.avg("peak_hour_latency_ms").alias("avg_peak_latency_ms"),
        F.avg("off_peak_latency_ms").alias("avg_offpeak_latency_ms"),
        F.avg("packet_loss_percent").alias("avg_packet_loss_pct"),
        F.avg("connection_drops").alias("avg_connection_drops"),
        F.sum(F.col("outage_flag").cast("int")).alias("outage_days"),
        F.sum(F.col("sla_breach").cast("int")).alias("sla_breach_days")
    )
)

# -----------------------------
# 5. Support ticket features (with resolution_days via DimDate)
# -----------------------------

# Prepare two views of DimDate for join: one for open, one for close
open_date_dim = dim_date_df.select(
    F.col("date_key").alias("open_date_key"),
    F.col("date").alias("open_date")
)

close_date_dim = dim_date_df.select(
    F.col("date_key").alias("close_date_key"),
    F.col("date").alias("close_date")
)

# Join support_tickets to DimDate to get real dates
tickets_with_dates = (
    tickets_df
    .join(open_date_dim, on="open_date_key", how="left")
    .join(close_date_dim, on="close_date_key", how="left")
    .withColumn(
        "resolution_days",
        F.when(
            (F.col("resolved") == True) & F.col("close_date").isNotNull(),
            F.datediff(F.col("close_date"), F.col("open_date"))
        )
    )
)

tickets_feats = (
    tickets_with_dates
    .groupBy("customer_key")
    .agg(
        F.count("*").alias("ticket_count"),
        F.avg(F.col("resolved").cast("int")).alias("ticket_resolved_rate"),
        F.avg("satisfaction_score").alias("avg_ticket_satisfaction"),
        F.avg("resolution_days").alias("avg_resolution_days")
    )
)

# -----------------------------
# 6. Billing features (with DimPaymentStatus)
# -----------------------------

# Join billing_events with DimPaymentStatus to get descriptive status
billing_with_status = (
    billing_df
    .join(
        dim_payment_status_df.select("payment_status_key", "payment_status"),
        on="payment_status_key",
        how="left"
    )
)

billing_feats = (
    billing_with_status
    .groupBy("customer_key")
    .agg(
        F.count("*").alias("billing_cycles"),
        F.avg("amount_charged_aud").alias("avg_amount_charged"),
        F.avg(F.when(F.col("payment_status") == "late",   1).otherwise(0)).alias("late_rate"),
        F.avg(F.when(F.col("payment_status") == "failed", 1).otherwise(0)).alias("failed_rate")
    )
)

# -----------------------------
# 7. Join all feature sets
# -----------------------------
# (Assuming you already defined cust_feats, telemetry_feats, tickets_feats above)

churn_features_df = (
    cust_feats
    .join(telemetry_feats, on="customer_key", how="left")
    .join(tickets_feats,   on="customer_key", how="left")
    .join(billing_feats,   on="customer_key", how="left")
)


In [0]:
churn_features_df.display()

customer_key,region_key,customer_id,customer_type,plan_type,contract_type,monthly_charge_aud,tenure_months,has_cybersecurity_addon,has_voip_addon,churn_flag,churn_date,region,avg_daily_bandwidth_gb,avg_peak_latency_ms,avg_offpeak_latency_ms,avg_packet_loss_pct,avg_connection_drops,outage_days,sla_breach_days,ticket_count,ticket_resolved_rate,avg_ticket_satisfaction,avg_resolution_days,billing_cycles,avg_amount_charged,late_rate,failed_rate
24,6,CUST-00024,business,NBN1000,2-year,129.4,11,true,false,false,null,Sydney,89.60922222222223,13.909999999999998,7.815555555555555,0.483,0.5444444444444444,3,2,3,1.0,2.6666666666666665,8.333333333333334,12,129.67416666666665,0.0,0.16666666666666666
30,6,CUST-00030,residential,NBN100,2-year,75.97,34,false,false,false,null,Sydney,18.313444444444446,12.934444444444443,7.395555555555554,0.433,0.4444444444444444,3,3,1,0.0,null,null,12,75.13000000000001,0.08333333333333333,0.08333333333333333
31,5,CUST-00031,wholesale,Wholesale_Dark_Fibre,2-year,934.22,60,false,false,false,null,Perth,394.51377777777765,22.308888888888887,12.305555555555555,0.35,0.25555555555555554,2,2,1,0.0,null,null,12,934.4775,0.16666666666666666,0.0
33,1,CUST-00033,residential,NBN50,1-year,58.32,60,false,false,false,null,Adelaide,8.082666666666666,19.74666666666667,11.00888888888889,0.3701111111111111,0.43333333333333335,2,1,1,1.0,2.0,6.0,12,58.5,0.16666666666666666,0.08333333333333333
45,6,CUST-00045,wholesale,Wholesale_Dark_Fibre,1-year,895.39,2,false,false,false,null,Sydney,404.0531111111111,12.607777777777775,6.883333333333332,0.4893333333333334,0.5111111111111111,3,3,2,1.0,2.0,7.0,12,895.3566666666666,0.0,0.16666666666666666
50,5,CUST-00050,business,Business_VoIP,2-year,146.89,41,false,true,false,null,Perth,5.215111111111112,24.282222222222217,13.338888888888889,0.39444444444444443,0.4222222222222222,2,2,1,0.0,null,null,12,146.24249999999998,0.0,0.0
52,2,CUST-00052,residential,NBN100,1-year,78.12,10,false,false,false,null,Brisbane,18.57344444444444,14.405555555555555,7.861111111111111,0.4106666666666667,0.3333333333333333,2,2,2,1.0,2.5,11.5,12,77.78833333333333,0.08333333333333333,0.0
57,6,CUST-00057,business,Business_VoIP,2-year,148.7,13,true,true,false,null,Sydney,5.072888888888889,13.782222222222225,7.564444444444446,0.3554444444444444,0.43333333333333335,3,2,1,1.0,3.0,14.0,12,148.01166666666666,0.25,0.08333333333333333
64,4,CUST-00064,business,Business_VoIP,1-year,149.9,15,false,true,false,null,Melbourne,4.949777777777779,12.416666666666666,6.731111111111111,0.4684444444444444,0.4666666666666667,3,3,1,1.0,5.0,4.0,12,148.99583333333337,0.16666666666666666,0.08333333333333333
68,3,CUST-00068,residential,NBN250,2-year,97.6,1,false,false,false,null,Canberra,41.605999999999995,10.32,5.557777777777778,0.23600000000000002,0.3111111111111111,0,0,1,0.0,null,null,12,96.99750000000002,0.16666666666666666,0.0


# Save the Dataset

In [0]:
# -----------------------------
# 8. Save the dataset
# -----------------------------

churn_features_final = churn_features_df.select(
    # IDs / labels for interpretation
    "customer_id",
    "region",
    
    # customer/commercial features
    "customer_type",
    "plan_type",
    "contract_type",
    "monthly_charge_aud",
    "tenure_months",
    "has_cybersecurity_addon",
    "has_voip_addon",
    
    # telemetry features
    "avg_daily_bandwidth_gb",
    "avg_peak_latency_ms",
    "avg_offpeak_latency_ms",
    "avg_packet_loss_pct",
    "avg_connection_drops",
    "outage_days",
    "sla_breach_days",
    
    # support features
    "ticket_count",
    "ticket_resolved_rate",
    "avg_ticket_satisfaction",
    "avg_resolution_days",
    
    # billing features
    "billing_cycles",
    "avg_amount_charged",
    "late_rate",
    "failed_rate",
    
    # label
    "churn_flag",
    "churn_date"
)

(
    churn_features_final
    .write
    .mode("overwrite")
    .format("delta")
    .save(CHURN_FEATURE_PATH)
)